# Data feasibility and quality

Real published results from DGT, INE and Eurostat. Author: Javier Saguar.

See [methodology](../docs/methodology.md), [sources](../docs/sources.md) and [limitations](../docs/limitations.md).

Reusable implementations live in `src/`; reproduce artifacts using the CLI before rerunning these explorations.

In [1]:
from pathlib import Path
import json
import pandas as pd
from ebdi.metrics.index import composite
from ebdi.utils.io import read_yaml
ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
TABLES = ROOT / "outputs/tables"
spain = pd.read_csv(TABLES / "spain_metrics.csv", dtype={"province_code": str})


## What observations exist?

In [2]:
quality = json.loads((TABLES / "data_quality.json").read_text())
assert quality["status"] == "passed"
print(pd.DataFrame([{k: r[k] for k in ["year", "rows", "fields", "fatalities_30d", "unknown_municipality", "unknown_collision"]} for r in quality["accidents"]]).to_string(index=False))
print("Missing exposures:", quality["missing_exposures"])

 year   rows  fields  fatalities_30d  unknown_municipality  unknown_collision
 2022  97916      73            1746                 11513                  0
 2023 101306      73            1806                 12201                  0
 2024 101996      73            1785                 12232                 35
Missing exposures: {'population': 0, 'registered_vehicles': 0, 'licensed_drivers': 0}


## Does every exposure join have the expected grain?

In [3]:
assert not spain.duplicated(["province_code", "year"]).any()
assert spain.groupby("year").size().eq(52).all()
assert spain.population.gt(0).all()
assert spain[["registered_vehicles", "licensed_drivers"]].notna().all().all()
print("156 unique province-years; all stock denominators observed")

156 unique province-years; all stock denominators observed


## Country-variable inclusion and actual publisher flags

In [4]:
metadata = pd.read_csv(TABLES / "country_comparability.csv").fillna("")
print(metadata.groupby(["variable", "included"]).size().to_string())
print(metadata.loc[metadata.status_flag.ne(""), ["geo", "year", "variable", "status_flag"]].head(15).to_string(index=False))

variable          included
fatalities        True        81
injury_crashes    False       81
insurance_claims  False       81
population        True        81
vehicle_km        False       81
geo  year   variable status_flag
 DE  2023 population           b
 FR  2023 population           p
 FR  2024 population           p
 PL  2022 population          ep
 PL  2023 population          ep
 RO  2022 population           e
 RO  2023 population           e
 RO  2024 population           e
